# LangGraph Workflow

Defines and compiles the **Stock Research StateGraph**.

## Execution topology

```
START
  │
  ▼
[planner]  ← normalizes ticker, initializes state
  │  │
  │  └──────────────────────┐
  ▼                         ▼
[news]  ← Tavily + GPT   [financials]  ← yfinance (no LLM)
  │                         │
  ▼                         │
[sentiment]  ← GPT score    │
  │                         │
  └──────────┬──────────────┘
             ▼
          [analyst]  ← gpt-4o synthesis
             │
             ▼
            END
```

**news** and **financials** run in parallel (both reachable from planner in the same superstep).  
**sentiment** waits for news. **analyst** waits for both sentiment + financials.

LangGraph tracks completion of all incoming edges before firing a node — automatic fan-in.

In [ ]:
import os
import sys

# Ensure project root is in path so agent notebooks are discoverable
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.basename(_here) != "graph" else os.path.dirname(_here)
if _root not in sys.path:
    sys.path.insert(0, _root)

import nbimporter  # noqa: E402 — enables importing .ipynb files as modules
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

# Import agent nodes and shared state from notebooks
from agents.planner import ResearchState, planner_node
from agents.news import news_node
from agents.financials import financials_node
from agents.sentiment import sentiment_node
from agents.analyst import analyst_node


def build_graph():
    """
    Build and compile the LangGraph StateGraph for stock research.

    Topology:
      planner → [news ∥ financials] → sentiment → analyst
    """
    builder = StateGraph(ResearchState)

    # Register nodes
    builder.add_node("planner", planner_node)
    builder.add_node("news", news_node)
    builder.add_node("financials", financials_node)
    builder.add_node("sentiment", sentiment_node)
    builder.add_node("analyst", analyst_node)

    # Wire edges
    builder.add_edge(START, "planner")

    # Fan-out: planner → news AND financials (parallel superstep)
    builder.add_edge("planner", "news")
    builder.add_edge("planner", "financials")

    # Sequential: news → sentiment (sentiment needs news_summary)
    builder.add_edge("news", "sentiment")

    # Fan-in: both sentiment and financials must complete before analyst
    builder.add_edge("sentiment", "analyst")
    builder.add_edge("financials", "analyst")

    builder.add_edge("analyst", END)

    return builder.compile()


# Compile once at import time (graph compilation has no API calls)
graph = build_graph()
print("[Workflow] Graph compiled successfully")


In [ ]:
if __name__ == "__main__":
    # --- Full end-to-end demo run ---
    import time

    TICKER = "AAPL"  # Change to NVDA, TSLA, MSFT, etc.

    print(f"\n{'='*60}")
    print(f"  Stock Research Agent — {TICKER}")
    print(f"{'='*60}\n")

    t0 = time.time()
    result = graph.invoke({"ticker": TICKER})
    elapsed = time.time() - t0

    print(f"\n{'='*60}")
    print(f"  REPORT  (completed in {elapsed:.1f}s)")
    print(f"{'='*60}\n")
    print(result["report"])
    print(f"\nSentiment Score: {result.get('sentiment_score', 'N/A')}")


In [ ]:
if __name__ == "__main__":
    # --- Visualize the graph (requires graphviz) ---
    try:
        from IPython.display import Image, display
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception as e:
        print(f"Graph visualization unavailable: {e}")
        # Print ASCII representation instead
        print(graph.get_graph().print_ascii())
